# 🎙️ Hindi Audiobook TTS Pipeline
### Latin → Devanagari → Edge-TTS Audio + SRT
**Pipeline:** Your Latin-script Hindi text → Qwen3.5:27b converts to Devanagari + assigns voices/prosody → edge-tts generates audio → SRT subtitle file download

---
**⚠️ BEFORE RUNNING:**
- Runtime → Change runtime type → **T4 GPU**
- Qwen3.5:27b needs ~15GB VRAM — right at T4's limit. If it OOMs, the notebook auto-falls back to `qwen2.5:14b`
- Ollama model download takes **10–20 minutes** on first run
---

In [1]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Install all dependencies                          ║
# ╚══════════════════════════════════════════════════════════════╝
print('📦 Installing dependencies...')

!pip install -q edge-tts aiofiles requests
!sudo apt-get install -y -q ffmpeg pciutils

import subprocess, os, sys
result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
print('✅ ffmpeg:', result.stdout.split('\n')[0])

import edge_tts
print('✅ edge-tts ready')
print('\n🎉 All dependencies installed!')

📦 Installing dependencies...
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
The following additional packages will be installed:
  libpci3 pci.ids
The following NEW packages will be installed:
  libpci3 pci.ids pciutils
0 upgraded, 3 newly installed, 0 to remove and 37 not upgraded.
Need to get 343 kB of archives.
After this operation, 1,581 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 pci.ids all 0.0~2022.01.22-1ubuntu0.1 [251 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpci3 amd64 1:3.7.0-6 [28.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 pciutils amd64 1:3.7.0-6 [63.6 kB]
Fetched 343 kB in 1s (637 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.

In [2]:


# Install and start Ollama server (required for Ollama models)
import subprocess, time, os, requests

print('🦙 Installing Ollama and Python library...')

!apt-get update -qq && apt-get install -y -qq zstd > /dev/null 2>&1
!pip install -q ollama
!curl -fsSL https://ollama.com/install.sh | sh

print('\n🚀 Starting Ollama server with memory optimizations...')

# ── Core settings ────────────────────────────────────────────────
os.environ['OLLAMA_HOST']          = '127.0.0.1:11434'
os.environ['OLLAMA_KEEP_ALIVE']    = '24h'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# ── VRAM optimizations (MUST be set before ollama serve) ─────────
# qwen3.5:27b = 17GB weights, T4 = 16GB VRAM → tight fit.
# Ollama auto-offloads overflow layers to CPU RAM.
# These two env vars cut the KV cache memory in half:
os.environ['OLLAMA_FLASH_ATTENTION'] = '1'      # enables Flash Attention (prerequisite for KV quant)
os.environ['OLLAMA_KV_CACHE_TYPE']   = 'q8_0'  # quantize KV cache: halves its VRAM usage

subprocess.Popen(
    ['/usr/local/bin/ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Robust polling: wait until server actually accepts HTTP connections
print('  Waiting for server...', end='')
server_ready = False
for _i in range(60):
    try:
        r = requests.get('http://127.0.0.1:11434/', timeout=2)
        if r.status_code == 200:
            server_ready = True
            break
    except Exception:
        pass
    print('.', end='', flush=True)
    time.sleep(1)

print()
if not server_ready:
    raise RuntimeError('❌ Ollama server did not start within 60s. Re-run this cell.')

print('✅ Ollama server running with Flash Attention + q8_0 KV cache!')
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader 2>/dev/null || echo 'No GPU — CPU mode'


🦙 Installing Ollama and Python library...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

🚀 Starting Ollama server with memory optimizations...
  Waiting for server....
✅ Ollama server running with Flash Attention + q8_0 KV cache!
Tesla T4, 15360 MiB, 14911 MiB


In [3]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Pull model + warm it into VRAM                   ║
# ╚══════════════════════════════════════════════════════════════╝
import subprocess, time
from ollama import chat as ollama_chat

PRIMARY_MODEL  = 'qwen3.5:27b'
FALLBACK_MODEL = 'qwen2.5:14b'
ACTIVE_MODEL   = None

def pull_model(model_name):
    print(f'📥 Pulling {model_name}... (10-20 min first run)')
    start = time.time()
    result = subprocess.run(['ollama', 'pull', model_name],
                            capture_output=True, text=True)
    elapsed = int(time.time() - start)
    if result.returncode == 0:
        print(f'✅ {model_name} pulled in {elapsed}s')
        return True
    print(f'❌ Pull failed: {result.stderr[-300:]}')
    return False


def warmup_model(model_name):
    """
    Force Ollama to load model weights into VRAM now, before Cell 6.

    KEY LESSONS from research:
    - Qwen3.5 uses ollama.chat(), NOT ollama.generate()
    - ollama.generate() returns chunk['response'] which is ALWAYS '' for
      thinking models — the answer lives in chunk.message.content
    - think=False is a TOP-LEVEL param to chat(), not inside options{}
    - Qwen3.5 does not support /no_think prompt switch (only Qwen3 does)
    """
    print(f'🔥 Warming up {model_name} into VRAM (may take 3-5 min first time)...')
    start = time.time()
    try:
        response_text = ''
        # Must use ollama.chat() — Qwen3.5 returns answer in message.content
        # think=False is TOP-LEVEL arg, disables hidden reasoning tokens
        stream = ollama_chat(
            model=model_name,
            messages=[{'role': 'user', 'content': 'Reply with one word: READY'}],
            think=False,         # ← TOP-LEVEL, not inside options{}
            stream=True,
            options={
                'num_predict': 5,
                'temperature': 1,
                'top_k': 20,
                'top_p': 0.95,
                'presence_penalty': 1.5,
            },
        )
        for chunk in stream:
            # chat() → answer is in chunk.message.content (NOT chunk['response'])
            response_text += chunk.message.content or ''
        elapsed = int(time.time() - start)
        print(f'✅ Model warm! Loaded into VRAM in {elapsed}s')
        print(f'   Warm-up response: "{response_text.strip()}"')
        return True
    except Exception as e:
        print(f'❌ Warm-up failed: {e}')
        return False


# ── Pull model ────────────────────────────────────────────────────
if pull_model(PRIMARY_MODEL):
    ACTIVE_MODEL = PRIMARY_MODEL
else:
    print(f'⚠️  Trying fallback: {FALLBACK_MODEL}')
    if pull_model(FALLBACK_MODEL):
        ACTIVE_MODEL = FALLBACK_MODEL
    else:
        raise RuntimeError('❌ Could not pull any model. Check your internet connection.')

print(f'\n🤖 Active model: {ACTIVE_MODEL}')

# ── Warm-up: load weights into VRAM before Cell 6 runs ───────────
if not warmup_model(ACTIVE_MODEL):
    print('⚠️  Warm-up failed but continuing. Cell 6 first call may be slow.')

print(f'\n✅ Model ready. Proceed to Cell 4.')


📥 Pulling qwen3.5:27b... (10-20 min first run)
✅ qwen3.5:27b pulled in 173s

🤖 Active model: qwen3.5:27b
🔥 Warming up qwen3.5:27b into VRAM (may take 3-5 min first time)...
✅ Model warm! Loaded into VRAM in 192s
   Warm-up response: "READY"

✅ Model ready. Proceed to Cell 4.


In [4]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Paste your Latin-script Hindi text here          ║
# ╚══════════════════════════════════════════════════════════════╝

# ─── PASTE YOUR TEXT BETWEEN THE TRIPLE QUOTES ────────────────
INPUT_TEXT = """
Ek din aisa hua tha.
Buddha jannat mein kamal taal ke kinare akele chal rahe the.
Taal mein khilte hue kamal bilkul jade ki tarah safed the, aur unke sone ke kesar se ek dilkash khushboo hawa mein fail rahi thi.
Lagta hai jannat mein subah ho rahi thi.
Thodi der baad, Buddha taal ke kinare ruk gaye aur kamal ke patton ko chhan kar neeche dekhne lage.
Yeh kamal taal seedhe narak ki gehrayi par bana hua tha, aur saaf paani se unhone telescope jaise River of Three Crossings (teen crossing wali nadi) aur Mountain of Needles (kaanthon ka pahad) dekha.
Phir unhone Kandata naam ke ek aadmi ko narak mein doosron ke saath tadapte hue dekh liya.
Yeh Kandata, ek badnaam chor tha jisne har tarah ki buraiyan ki thi – logon ko maara aur gharon ko aag laga di.
Lekin uske khate mein ek achha kaam bhi tha.
Ek baar woh jungle se guzarte waqt usko ek chhota makdi raste par chalte hue dikha.
Kandata use machalane wala hi tha, lekin phir socha,
“Nahi yaar, yeh bhi toh jaan hai,”
aur usne usse jaane diya.
Buddha ko yeh yaad aa gaya aur unhone socha,
“Main iski madad kar sakta hoon.”
To unhone jannat ke kamal taal mein ek makdi dhunda aur uska dhaaga narak tak utaar diya.
Makdi ka dhaaga patla tha lekin mazboot, aur seedha narak mein chala gaya.
Kandata, jo narak mein dard se karah raha tha, usne achanak woh dhaaga dekha.
“Agar main is par charhunga toh pakka narak se bahar aa jaaunga,”
usne socha aur poori taqat se dhaage par chadhna shuru kar diya.
Lekin jab woh thoda upar chadha, to neeche dekh ke usko doosre paapi bhi wahi dhaaga charhte hue dikhe.
Yeh dekhte hi Kandata chillaya,
“Yeh makdi ka dhaaga mera hai!
Utar jao! Utar jao!”
Us pal hi, *plink* ki awaaz ke saath makdi ka dhaaga toot gaya.
Kandata aur doosre paapi sab narak mein gir gaye.
Jannat ke kamal taal ke kinare khade Buddha ne yeh dekha, unke chehre par udaasi thi, aur phir woh chupchap chalne lage.
"""
# ──────────────────────────────────────────────────────────────

# Book/Chapter metadata (used in output filename)
BOOK_NAME    = "My_Audiobook"
CHAPTER_NAME = "Chapter_01"

print(f'📖 Input text loaded: {len(INPUT_TEXT)} characters')
print('\nPreview (first 200 chars):')
print(INPUT_TEXT[:200].strip() + '...')

📖 Input text loaded: 1876 characters

Preview (first 200 chars):
Ek din aisa hua tha.
Buddha jannat mein kamal taal ke kinare akele chal rahe the.
Taal mein khilte hue kamal bilkul jade ki tarah safed the, aur unke sone ke kesar se ek dilkash khushboo hawa mein fa...


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Edge-TTS voices, prosody presets, master prompt  ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Voice assignments ─────────────────────────────────────────────
# narrator_male removed: it caused confusion — Qwen would emit 'narrator_male'
# which is NOT a valid character type. All narration uses 'narrator'.
VOICES = {
    'narrator'    : 'hi-IN-SwaraNeural',   # default narrator — all non-dialogue text
    'char_male'   : 'hi-IN-MadhurNeural',  # only for quoted male speech
    'char_female' : 'hi-IN-SwaraNeural',   # only for quoted female speech
    'char_child'  : 'hi-IN-SwaraNeural',   # only for quoted child speech
    'char_elder'  : 'hi-IN-MadhurNeural',  # only for quoted elder speech
    'inner_thought': 'hi-IN-SwaraNeural',  # only for quoted inner thoughts
}

# ── Prosody presets (rate, pitch, volume) ─────────────────────────
# Split into NARRATOR emotions and CHARACTER emotions.
# Narrator MUST only use narrator emotions. Characters use character emotions.
PRESETS = {
    # ── NARRATOR ONLY emotions ────────────────────────────────────
    'narrator_calm'  : ('+0%',   '+0Hz',   '+0%'),   # default neutral narration
    'narrator_slow'  : ('-15%',  '-5Hz',   '+0%'),   # dramatic build-up, important reveal
    'tension'        : ('+8%',   '+10Hz',  '+5%'),   # suspense building, danger approaching
    'relief'         : ('-8%',   '-5Hz',   '+0%'),   # resolution, safety restored

    # ── CHARACTER SPEECH emotions ─────────────────────────────────
    'dialogue_normal': ('+5%',   '+0Hz',   '+0%'),   # regular calm conversation
    'excited'        : ('+30%',  '+40Hz',  '+15%'),  # joy, discovery, surprise
    'sad'            : ('-30%',  '-40Hz',  '-15%'),  # grief, loss, disappointment
    'angry'          : ('+20%',  '+25Hz',  '+20%'),  # rage, confrontation
    'whisper'        : ('-20%',  '-25Hz',  '-35%'),  # secret, danger nearby
    'scared'         : ('+25%',  '+30Hz',  '-10%'),  # fear, horror
    'sarcastic'      : ('-15%',  '-15Hz',  '+0%'),   # irony, dry humor
    'child_happy'    : ('+15%',  '+80Hz',  '+10%'),  # playful child
    'elder_wise'     : ('-20%',  '-60Hz',  '+0%'),   # slow, deliberate elder
    'inner_thought'  : ('-12%',  '-15Hz',  '-20%'),  # quiet, reflective thought
    'humorous'       : ('+10%',  '+20Hz',  '+10%'),  # funny, lighthearted
}

# Valid emotion sets per character type — used in Cell 7 validation
NARRATOR_EMOTIONS  = {'narrator_calm', 'narrator_slow', 'tension', 'relief'}
CHARACTER_EMOTIONS = {'dialogue_normal', 'excited', 'sad', 'angry', 'whisper',
                      'scared', 'sarcastic', 'child_happy', 'elder_wise',
                      'inner_thought', 'humorous'}

# ── MASTER SYSTEM PROMPT ──────────────────────────────────────────
SYSTEM_PROMPT = """You are an expert Hindi audiobook director and TTS script engineer. You receive Hindi story text (in Latin script) and convert it into a precise JSON script for edge-tts narration.

═══════════════════════════════════════════════════════════════
STEP 1: LANGUAGE CONVERSION
═══════════════════════════════════════════════════════════════
Convert ALL Latin-script Hindi/Hinglish to Devanagari.
Examples: "ek" → "एक", "bahut" → "बहुत", "chal rahe the" → "चल रहे थे"

KEEP these in original English (edge-tts pronounces them better):
• Brand names: Google, YouTube, WhatsApp, Amazon, Microsoft
• Tech terms: AI, TTS, WiFi, app, laptop, computer, internet
• Common English words used in Indian speech: "interesting", "actually", "curious"
• Proper nouns that are universally known in English

Numbers → Devanagari words: "3" → "तीन", "1947" → "उन्नीस सौ सैंतालीस"

═══════════════════════════════════════════════════════════════
STEP 2: CHARACTER ASSIGNMENT — THE MOST IMPORTANT RULE
═══════════════════════════════════════════════════════════════
Apply this 3-step test to EVERY sentence:

  TEST: Is this text inside quotation marks ("...", '...', or "...") ?
  ├─ NO  → character: "narrator"   (MANDATORY — no exceptions)
  └─ YES → Identify the speaker:
           ├─ Male character speaking    → "char_male"
           ├─ Female character speaking  → "char_female"
           ├─ Child speaking             → "char_child"
           ├─ Elder (60+) speaking       → "char_elder"
           └─ Someone THINKING (quoted)  → "inner_thought"

⚠️  CRITICAL RULES:
• A sentence describing or narrating a character is STILL narration → "narrator"
• "Buddha walked", "Kandata saw", "She ran" — ALL narrator, even if about a character
• ONLY the actual quoted words get a character type
• When unsure, ALWAYS default to "narrator"
• "inner_thought" is ONLY for quoted thoughts (ne socha "..."), NOT for the narrating sentence

═══════════════════════════════════════════════════════════════
STEP 3: EMOTION ASSIGNMENT
═══════════════════════════════════════════════════════════════
Narrator segments → use ONLY narrator emotions:
  "narrator_calm"  → scene description, normal narration (default)
  "narrator_slow"  → dramatic build-up, important revelation, scene change
  "tension"        → danger approaching, suspense, something bad about to happen
  "relief"         → resolution, safety, peace restored

Character segments → use character emotions:
  "dialogue_normal" → calm, regular speech
  "excited"         → joy, surprise, triumph
  "sad"             → grief, regret, loss
  "angry"           → rage, frustration, confrontation
  "whisper"         → secret, danger, intimacy
  "scared"          → fear, horror, panic
  "sarcastic"       → irony, teasing
  "child_happy"     → playful child energy
  "elder_wise"      → slow, deliberate elder wisdom
  "inner_thought"   → reflective, quiet, personal
  "humorous"        → funny, lighthearted

⚠️  NEVER assign "child_happy", "elder_wise", "sarcastic" to narrator segments.
⚠️  NEVER assign "narrator_calm", "narrator_slow", "tension", "relief" to character segments.

═══════════════════════════════════════════════════════════════
STEP 4: PAUSE ASSIGNMENT (pause_after in milliseconds)
═══════════════════════════════════════════════════════════════
800ms  → after paragraph break, after scene change
600ms  → end of long narration sentence, after important reveal
500ms  → after "!" or "?" in speech
400ms  → normal end of sentence
300ms  → between short action lines
200ms  → after dialogue tag ("usne kaha,"), before the quoted speech begins
700ms  → after dramatic "..." pause in text

═══════════════════════════════════════════════════════════════
STEP 5: SEGMENTATION RULES
═══════════════════════════════════════════════════════════════
• Max 2 sentences per segment (shorter = better prosody)
• Split narrator introduction from the quote it introduces:
  "Usne kaha," → narrator segment (pause_after: 200)
  "Yeh mera hai!" → char_male segment (pause_after: 500)
• Never merge narrator text with dialogue in the same segment
• Never cut a sentence in half

═══════════════════════════════════════════════════════════════
FEW-SHOT EXAMPLE — Study this carefully
═══════════════════════════════════════════════════════════════
Input text:
"Raghu jungle mein chal raha tha. Achanak usne ek sher dekha. Woh dar gaya aur socha, 'Mujhe bhagna chahiye.' Phir woh tez bhaaga."

Correct output:
[
  {"id": 1, "character": "narrator", "emotion": "narrator_calm", "text": "रघु जंगल में चल रहा था।", "pause_after": 400},
  {"id": 2, "character": "narrator", "emotion": "tension", "text": "अचानक उसने एक शेर देखा।", "pause_after": 600},
  {"id": 3, "character": "narrator", "emotion": "tension", "text": "वह डर गया और सोचा,", "pause_after": 200},
  {"id": 4, "character": "inner_thought", "emotion": "scared", "text": "मुझे भागना चाहिए।", "pause_after": 500},
  {"id": 5, "character": "narrator", "emotion": "tension", "text": "फिर वह तेज़ भागा।", "pause_after": 800}
]

WHY: "Raghu jungle mein chal raha tha" → narration about Raghu → narrator
     "Achanak usne ek sher dekha" → narration about what he saw → narrator + tension emotion
     "Woh dar gaya aur socha," → narration introducing thought → narrator (short pause before thought)
     "'Mujhe bhagna chahiye'" → QUOTED thought → inner_thought + scared
     "Phir woh tez bhaaga" → narration again → narrator + tension still
═══════════════════════════════════════════════════════════════
OUTPUT FORMAT
═══════════════════════════════════════════════════════════════
Output ONLY a raw JSON array. No markdown, no code fences, no explanation.
Start directly with [ and end with ]

Each object:
{
  "id": <integer>,
  "character": <one of: "narrator","char_male","char_female","char_child","char_elder","inner_thought">,
  "emotion": <see allowed emotions above>,
  "text": <Devanagari text only>,
  "pause_after": <integer milliseconds>
}
"""

print("✅ VOICES, PRESETS, SYSTEM_PROMPT defined")
print(f"   Narrator emotions : {len(NARRATOR_EMOTIONS)}")
print(f"   Character emotions: {len(CHARACTER_EMOTIONS)}")
print(f"   Voice roles       : {len(VOICES)}")
print(f"   Prosody presets   : {len(PRESETS)}")
print(f"   System prompt     : {len(SYSTEM_PROMPT)} chars")



In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Send text to Qwen3.5, get structured script      ║
# ╚══════════════════════════════════════════════════════════════╝
import json, re, time
from ollama import chat as ollama_chat

# ── Pydantic schema for guaranteed valid JSON output ──────────────
# This passes a JSON schema to Ollama so it CONSTRAINS the model to
# only emit valid enum values for character and emotion — no hallucinated
# values, no invalid types, always parseable JSON.
try:
    from pydantic import BaseModel, RootModel
    from typing import Literal, List
    PYDANTIC_OK = True

    class Segment(BaseModel):
        id: int
        character: Literal[
            'narrator', 'char_male', 'char_female',
            'char_child', 'char_elder', 'inner_thought'
        ]
        emotion: Literal[
            'narrator_calm', 'narrator_slow', 'tension', 'relief',
            'dialogue_normal', 'excited', 'sad', 'angry', 'whisper',
            'scared', 'sarcastic', 'child_happy', 'elder_wise',
            'inner_thought', 'humorous'
        ]
        text: str
        pause_after: int

    class Script(RootModel):
        root: List[Segment]

    SCHEMA = Script.model_json_schema()
    print("✅ Pydantic schema loaded — Ollama will enforce valid enum values")

except ImportError:
    PYDANTIC_OK = False
    SCHEMA = None
    print("⚠️  Pydantic not found — install with: pip install pydantic")
    print("   Falling back to regex JSON extraction")


def call_qwen(text_chunk, system_prompt, model, chunk_index=0, total_chunks=1):
    """Call Qwen3.5 with schema enforcement and narrator-bias instructions."""
    full_content = []

    # Prepend a reminder for continuity across chunks
    context_hint = ""
    if total_chunks > 1:
        context_hint = (
            f"[Chunk {chunk_index+1} of {total_chunks}. "
            "Maintain consistent narrator voice for all non-quoted text. "
            "Only quoted speech/thoughts get character types.]\n\n"
        )

    stream = ollama_chat(
        model=model,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {
                'role': 'user',
                'content': (
                    context_hint +
                    'Convert the following text to the JSON script format. '
                    'Remember: ALL non-quoted text = narrator. '
                    'Output ONLY the raw JSON array starting with [\n\n'
                    + text_chunk
                )
            },
        ],
        think=False,
        stream=True,
        format=SCHEMA,          # ← schema enforcement: constrains valid values
        options={
            'temperature'     : 1,
            'top_k'           : 20,
            'top_p'           : 0.95,
            'presence_penalty': 1.5,
        },
    )
    for chunk in stream:
        full_content.append(chunk.message.content or '')
    return ''.join(full_content)


def extract_json(raw_text):
    """Extract and validate JSON array from model output."""
    raw_text = re.sub(r'<think>[\s\S]*?</think>', '', raw_text, flags=re.IGNORECASE)
    raw_text = re.sub(r'```json\s*', '', raw_text)
    raw_text = re.sub(r'```\s*', '', raw_text)
    raw_text = raw_text.strip()

    start = raw_text.find('[')
    end   = raw_text.rfind(']') + 1
    if start == -1 or end == 0:
        raise ValueError(f'No JSON array found.\nRaw: {raw_text[:400]}')

    json_str = raw_text[start:end]
    try:
        return json.loads(json_str)
    except json.JSONDecodeError as e:
        print(f'  ⚠️  JSON parse error ({e}), recovering...')
        last_brace = json_str.rfind('}')
        if last_brace != -1:
            return json.loads(json_str[:last_brace + 1] + ']')
        raise


def auto_correct_emotions(segments):
    """
    Post-processing: enforce emotion-to-character consistency.
    This catches any cases where Qwen3.5 still assigns wrong emotion
    types despite the schema/prompt constraints.
    """
    corrections = 0
    for seg in segments:
        char = seg.get('character', 'narrator')
        emo  = seg.get('emotion', 'narrator_calm')

        if char == 'narrator' and emo not in NARRATOR_EMOTIONS:
            seg['emotion'] = 'narrator_calm'
            corrections += 1
        elif char != 'narrator' and emo not in CHARACTER_EMOTIONS:
            seg['emotion'] = 'dialogue_normal'
            corrections += 1

    if corrections:
        print(f'  🔧 Auto-corrected {corrections} emotion mismatch(es)')
    return segments


def chunk_text(text, max_chars=1000):
    """Split at paragraph/line boundaries. 1000 chars ≈ 250 tokens."""
    paragraphs = [p.strip() for p in text.strip().split('\n') if p.strip()]
    chunks, current = [], ''
    for para in paragraphs:
        if len(current) + len(para) > max_chars and current:
            chunks.append(current.strip())
            current = para
        else:
            current += '\n' + para
    if current.strip():
        chunks.append(current.strip())
    return chunks


# ── Main conversion loop ──────────────────────────────────────────
text_chunks = chunk_text(INPUT_TEXT, max_chars=1000)
total       = len(text_chunks)
print(f'📑 Text split into {total} chunk(s)')
print(f'   Schema enforcement: {"ON ✅" if PYDANTIC_OK else "OFF ⚠️"}')

all_segments, segment_id = [], 1

for i, chunk in enumerate(text_chunks):
    print(f'\n🤖 Chunk {i+1}/{total} ({len(chunk)} chars)...')
    start_t = time.time()
    raw = ''
    try:
        raw = call_qwen(chunk, SYSTEM_PROMPT, ACTIVE_MODEL,
                        chunk_index=i, total_chunks=total)
        if not raw.strip():
            raise ValueError('Empty response — re-run Cell 3 to reload model')
        segments = extract_json(raw)
        segments = auto_correct_emotions(segments)
        for seg in segments:
            seg['id'] = segment_id
            segment_id += 1
        all_segments.extend(segments)
        print(f'  ✅ {len(segments)} segments in {time.time()-start_t:.1f}s')
    except Exception as e:
        print(f'  ❌ Error on chunk {i+1}: {e}')
        if raw:
            print(f'  Raw output (first 500 chars):\n  {raw[:500]}')
        raise

print(f'\n✅ Total segments: {len(all_segments)}')

# ── Narrator consistency report ───────────────────────────────────
char_counts = {}
for s in all_segments:
    char_counts[s.get('character','?')] = char_counts.get(s.get('character','?'), 0) + 1

print('\n📊 Character breakdown:')
for char, count in sorted(char_counts.items(), key=lambda x: -x[1]):
    pct = 100 * count / len(all_segments)
    print(f'   {char:<16}: {count:3d} segments ({pct:.0f}%)')

print('\n📋 First 8 segments:')
for seg in all_segments[:8]:
    print(f'  [{seg["id"]:02d}] {seg["character"]:<14} | {seg["emotion"]:<16} | {seg["text"][:55]}')



In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Validate & map segments to TTS parameters        ║
# ╚══════════════════════════════════════════════════════════════╝

def resolve_voice(character):
    """Map character role to edge-tts voice name."""
    mapping = {
        'narrator'      : VOICES['narrator'],
        'char_male'     : VOICES['char_male'],
        'char_female'   : VOICES['char_female'],
        'char_child'    : VOICES['char_child'],
        'char_elder'    : VOICES['char_elder'],
        'inner_thought' : VOICES['inner_thought'],
    }
    return mapping.get(character, VOICES['narrator'])

def resolve_prosody(emotion, character):
    """Return (rate, pitch, volume) for an emotion+character combo."""
    base = PRESETS.get(emotion, PRESETS['narrator_calm'])
    rate, pitch, volume = base

    # Extra pitch adjustments for character types
    if character == 'char_child':
        # Shift pitch higher for child voices
        pitch_val = int(pitch.replace('Hz', '').replace('+', ''))
        pitch_val = min(pitch_val + 60, 180)
        pitch = f'+{pitch_val}Hz' if pitch_val >= 0 else f'{pitch_val}Hz'

    if character == 'char_elder':
        pitch_val = int(pitch.replace('Hz', '').replace('+', ''))
        pitch_val = max(pitch_val - 30, -180)
        pitch = f'{pitch_val}Hz' if pitch_val < 0 else f'+{pitch_val}Hz'

    return rate, pitch, volume

# Validate and enrich segments
VALID_EMOTIONS    = set(PRESETS.keys())
VALID_CHARACTERS  = {'narrator', 'narrator_male', 'char_male', 'char_female',
                     'char_child', 'char_elder', 'inner_thought'}

enriched = []
issues   = 0

for seg in all_segments:
    # Sanitize
    if seg.get('character') not in VALID_CHARACTERS:
        seg['character'] = 'narrator'
        issues += 1
    if seg.get('emotion') not in VALID_EMOTIONS:
        seg['emotion'] = 'narrator_calm'
        issues += 1
    if not seg.get('text', '').strip():
        continue  # skip empty segments

    # Resolve voice + prosody
    voice              = resolve_voice(seg['character'])
    rate, pitch, vol   = resolve_prosody(seg['emotion'], seg['character'])
    pause              = int(seg.get('pause_after', 400))

    enriched.append({
        'id'          : seg['id'],
        'text'        : seg['text'].strip(),
        'character'   : seg['character'],
        'emotion'     : seg['emotion'],
        'voice'       : voice,
        'rate'        : rate,
        'pitch'       : pitch,
        'volume'      : vol,
        'pause_after' : pause,
    })

print(f'✅ Validated {len(enriched)} segments ({issues} field(s) auto-corrected)')
print('\n📋 Enriched segment sample:')
for s in enriched[:3]:
    print(f'  [{s["id"]:02d}] {s["character"]:<14} | {s["emotion"]:<16} | voice={s["voice"]}')
    print(f'       rate={s["rate"]:<7} pitch={s["pitch"]:<8} vol={s["volume"]}')
    print(f'       text: "{s["text"][:70]}"')
    print()

In [8]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Generate audio for all segments (parallel)       ║
# ╚══════════════════════════════════════════════════════════════╝
import asyncio, edge_tts, os, time

AUDIO_DIR   = '/content/audio_chunks'
os.makedirs(AUDIO_DIR, exist_ok=True)

MAX_CONCURRENT = 4   # parallel edge-tts requests
MAX_RETRIES    = 3

async def generate_segment_audio(seg, semaphore):
    """Generate audio for one segment, return (seg_id, mp3_path, duration_ms)."""
    out_path = os.path.join(AUDIO_DIR, f'seg_{seg["id"]:04d}.mp3')
    srt_path = os.path.join(AUDIO_DIR, f'seg_{seg["id"]:04d}.srt')

    async with semaphore:
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                comm = edge_tts.Communicate(
                    seg['text'],
                    seg['voice'],
                    rate   = seg['rate'],
                    pitch  = seg['pitch'],
                    volume = seg['volume'],
                )
                submaker = edge_tts.SubMaker()
                with open(out_path, 'wb') as f:
                    async for chunk in comm.stream():
                        if chunk['type'] == 'audio':
                            f.write(chunk['data'])
                        elif chunk['type'] in ('WordBoundary', 'SentenceBoundary'):
                            submaker.feed(chunk)

                # Save per-segment SRT
                with open(srt_path, 'w', encoding='utf-8') as f:
                    f.write(submaker.get_srt())

                # Get duration via ffprobe
                dur_proc = await asyncio.create_subprocess_exec(
                    'ffprobe', '-v', 'error', '-show_entries', 'format=duration',
                    '-of', 'default=noprint_wrappers=1:nokey=1', out_path,
                    stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.DEVNULL
                )
                dur_out, _ = await dur_proc.communicate()
                duration_ms = int(float(dur_out.decode().strip()) * 1000)

                return seg['id'], out_path, srt_path, duration_ms

            except Exception as e:
                if attempt == MAX_RETRIES:
                    print(f'  ❌ Seg {seg["id"]} FAILED after {MAX_RETRIES} attempts: {e}')
                    raise
                await asyncio.sleep(2 * attempt)

async def generate_all(segments):
    sem    = asyncio.Semaphore(MAX_CONCURRENT)
    tasks  = [generate_segment_audio(s, sem) for s in segments]
    total  = len(tasks)
    done   = 0
    results = []

    for coro in asyncio.as_completed(tasks):
        result = await coro
        results.append(result)
        done += 1
        seg_id = result[0]
        dur_s  = result[3] / 1000
        print(f'  ✓ [{done:03d}/{total}] seg_{seg_id:04d}  {dur_s:.1f}s', end='\r')

    print()  # newline after \r
    # Sort by segment ID to maintain order
    results.sort(key=lambda x: x[0])
    return results

print(f'🎙️  Generating audio for {len(enriched)} segments...\n')
start_time = time.time()

audio_results = await generate_all(enriched)

total_audio_ms = sum(r[3] for r in audio_results)
total_time     = time.time() - start_time
print(f'\n✅ All audio generated!')
print(f'   Total audio duration : {total_audio_ms/1000:.1f}s ({total_audio_ms/60000:.1f} min)')
print(f'   Generation time      : {total_time:.1f}s')
print(f'   Speed ratio          : {total_audio_ms/1000/total_time:.1f}x realtime')

🎙️  Generating audio for 24 segments...

  ✓ [024/24] seg_0004  3.6s

✅ All audio generated!
   Total audio duration : 150.8s (2.5 min)
   Generation time      : 9.0s
   Speed ratio          : 16.8x realtime


In [9]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 9 — Merge audio chunks + build master SRT            ║
# ╚══════════════════════════════════════════════════════════════╝
import subprocess, os, re
from pathlib import Path

OUTPUT_DIR  = '/content/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

FINAL_MP3 = os.path.join(OUTPUT_DIR, f'{BOOK_NAME}_{CHAPTER_NAME}.mp3')
FINAL_SRT = os.path.join(OUTPUT_DIR, f'{BOOK_NAME}_{CHAPTER_NAME}.srt')

# ── Step 1: Merge MP3 chunks with ffmpeg ──────────────────────────
concat_list = os.path.join(AUDIO_DIR, 'concat_list.txt')
with open(concat_list, 'w') as f:
    for seg_id, mp3_path, srt_path, dur_ms in audio_results:
        f.write(f"file '{mp3_path}'\n")

print('🔗 Merging audio chunks...')
merge_result = subprocess.run([
    'ffmpeg', '-y', '-f', 'concat', '-safe', '0',
    '-i', concat_list,
    '-acodec', 'copy',
    FINAL_MP3
], capture_output=True, text=True)

if merge_result.returncode != 0:
    print('❌ ffmpeg merge error:')
    print(merge_result.stderr[-500:])
    raise RuntimeError('Audio merge failed')

mp3_size = os.path.getsize(FINAL_MP3) // 1024
print(f'✅ Audio merged → {FINAL_MP3} ({mp3_size} KB)')

# ── Step 2: Build master SRT with cumulative timestamps ───────────
def ms_to_srt_time(ms):
    """Convert milliseconds to SRT timestamp: HH:MM:SS,mmm"""
    ms   = max(0, int(ms))
    h    = ms // 3600000; ms %= 3600000
    m    = ms // 60000;   ms %= 60000
    s    = ms // 1000;    ms %= 1000
    return f'{h:02d}:{m:02d}:{s:02d},{ms:03d}'

def parse_srt(srt_text):
    """Parse an SRT string into list of (start_ms, end_ms, text) tuples."""
    entries = []
    blocks  = re.split(r'\n\n+', srt_text.strip())
    for block in blocks:
        lines = block.strip().split('\n')
        if len(lines) < 3:
            continue
        # Line 0: index, Line 1: timestamps, Line 2+: text
        time_match = re.search(
            r'(\d{2}:\d{2}:\d{2}[,.]\d{3})\s*-->\s*(\d{2}:\d{2}:\d{2}[,.]\d{3})',
            lines[1]
        )
        if not time_match:
            continue
        def parse_ts(ts):
            ts = ts.replace(',', '.')
            h, m, rest = ts.split(':')
            s, ms = rest.split('.')
            return int(h)*3600000 + int(m)*60000 + int(s)*1000 + int(ms[:3])
        start_ms = parse_ts(time_match.group(1))
        end_ms   = parse_ts(time_match.group(2))
        text     = ' '.join(lines[2:])
        entries.append((start_ms, end_ms, text))
    return entries

# Build master SRT
master_entries = []
time_offset_ms = 0
srt_index      = 1

for seg_id, mp3_path, srt_path, dur_ms in audio_results:
    # Find matching enriched segment for character info
    seg_info = next((s for s in enriched if s['id'] == seg_id), {})
    char_tag = seg_info.get('character', 'narrator')

    # Parse per-segment SRT
    try:
        with open(srt_path, 'r', encoding='utf-8') as f:
            srt_content = f.read().strip()
    except:
        srt_content = ''

    if srt_content:
        entries = parse_srt(srt_content)
        for start_ms, end_ms, text in entries:
            global_start = time_offset_ms + start_ms
            global_end   = time_offset_ms + end_ms
            master_entries.append((
                srt_index,
                global_start,
                global_end,
                text
            ))
            srt_index += 1
    else:
        # Fallback: full segment text with estimated duration
        text = seg_info.get('text', '')
        if text:
            master_entries.append((
                srt_index,
                time_offset_ms,
                time_offset_ms + dur_ms,
                text
            ))
            srt_index += 1

    # Advance offset: audio duration + pause after
    pause_ms    = seg_info.get('pause_after', 400)
    time_offset_ms += dur_ms + pause_ms

# Write master SRT
with open(FINAL_SRT, 'w', encoding='utf-8') as f:
    for idx, start_ms, end_ms, text in master_entries:
        f.write(f'{idx}\n')
        f.write(f'{ms_to_srt_time(start_ms)} --> {ms_to_srt_time(end_ms)}\n')
        f.write(f'{text}\n\n')

srt_size = os.path.getsize(FINAL_SRT) // 1024
total_duration_str = ms_to_srt_time(time_offset_ms)

print(f'✅ SRT file built → {FINAL_SRT}')
print(f'   Subtitle entries : {len(master_entries)}')
print(f'   Total duration   : {total_duration_str}')
print(f'   File size        : {srt_size} KB')

# Preview first 5 SRT entries
print('\n📋 SRT Preview (first 5 entries):')
for entry in master_entries[:5]:
    idx, s, e, t = entry
    print(f'  {idx}  {ms_to_srt_time(s)} --> {ms_to_srt_time(e)}')
    print(f'  {t}')
    print()

🔗 Merging audio chunks...
✅ Audio merged → /content/output/My_Audiobook_Chapter_01.mp3 (883 KB)
✅ SRT file built → /content/output/My_Audiobook_Chapter_01.srt
   Subtitle entries : 26
   Total duration   : 00:02:48,992
   File size        : 4 KB

📋 SRT Preview (first 5 entries):
  1  00:00:00,100 --> 00:00:02,637
  एक दिन ऐसा हुआ था।

  2  00:00:03,292 --> 00:00:08,909
  बुद्ध स्वर्ग में कमल ताल के किनारे अकेले चल रहे थे।

  3  00:00:09,780 --> 00:00:19,342
  ताल में खिलते हुए कमल बिल्कुल जड़ की तरह सफेद थे, और उनके सोने के केशर से एक दिलकश खुशबू हवा में फैल रही थी।

  4  00:00:20,204 --> 00:00:23,729
  लगता है स्वर्ग में सुबह हो रही थी।

  5  00:00:24,580 --> 00:00:33,550
  थोड़ी देर बाद, बुद्ध ताल के किनारे रुक गए और कमल के पत्तों को छानकर नीचे देखने लगे।



In [10]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 10 — Preview audio in notebook + Download files      ║
# ╚══════════════════════════════════════════════════════════════╝
from IPython.display import Audio, display, HTML
from google.colab import files
import json

# ── Play audio directly in notebook ──────────────────────────────
print('🎧 Audio preview (first 30 seconds):')
display(Audio(FINAL_MP3, autoplay=False))

# ── Save the Qwen script as JSON too (useful for re-runs) ────────
SCRIPT_JSON = os.path.join(OUTPUT_DIR, f'{BOOK_NAME}_{CHAPTER_NAME}_script.json')
with open(SCRIPT_JSON, 'w', encoding='utf-8') as f:
    json.dump(enriched, f, ensure_ascii=False, indent=2)

print(f'\n📁 Output files:')
print(f'   🔊 Audio   : {FINAL_MP3}')
print(f'   📝 SRT     : {FINAL_SRT}')
print(f'   📋 Script  : {SCRIPT_JSON}')

# ── Download all three files ─────────────────────────────────────
print('\n⬇️  Downloading files to your computer...')
files.download(FINAL_SRT)    # SRT first — small file
files.download(SCRIPT_JSON)  # JSON script
files.download(FINAL_MP3)    # Audio — may take a moment

# ── Summary stats ────────────────────────────────────────────────
print('\n' + '='*60)
print('📊 PIPELINE SUMMARY')
print('='*60)
print(f'  Input  : {len(INPUT_TEXT)} chars of Latin-script Hindi')
print(f'  Model  : {ACTIVE_MODEL}')
print(f'  Segs   : {len(enriched)} segments')

char_counts = {}
for s in enriched:
    char_counts[s['character']] = char_counts.get(s['character'], 0) + 1
for char, count in sorted(char_counts.items(), key=lambda x: -x[1]):
    print(f'  {char:<20} : {count} segment(s)')

print(f'  Audio  : {os.path.getsize(FINAL_MP3)//1024} KB')
print(f'  SRT    : {len(master_entries)} subtitle lines')
print('='*60)
print('✅ Done!')

🎧 Audio preview (first 30 seconds):



📁 Output files:
   🔊 Audio   : /content/output/My_Audiobook_Chapter_01.mp3
   📝 SRT     : /content/output/My_Audiobook_Chapter_01.srt
   📋 Script  : /content/output/My_Audiobook_Chapter_01_script.json

⬇️  Downloading files to your computer...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


📊 PIPELINE SUMMARY
  Input  : 1876 chars of Latin-script Hindi
  Model  : qwen3.5:27b
  Segs   : 24 segments
  narrator             : 20 segment(s)
  char_male            : 2 segment(s)
  inner_thought        : 2 segment(s)
  Audio  : 883 KB
  SRT    : 26 subtitle lines
✅ Done!


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 11 — OPTIONAL: Re-run TTS from saved JSON script     ║
# ║  (Use this to re-generate audio without calling Qwen again) ║
# ╚══════════════════════════════════════════════════════════════╝

# Uncomment and run this cell if you want to re-generate audio
# from a previously saved script JSON (saves Qwen processing time):

# from google.colab import files
# import json
#
# print('Upload your saved _script.json file:')
# uploaded = files.upload()
# fname = list(uploaded.keys())[0]
# enriched = json.loads(uploaded[fname])
# print(f'Loaded {len(enriched)} segments from {fname}')
# print('Now run Cell 8 onwards to regenerate audio.')

print('This cell is optional. Uncomment the code above to use it.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 12 — OPTIONAL: Inspect/edit specific segments        ║
# ║  Tweak voice/emotion for a segment and regenerate it alone ║
# ╚══════════════════════════════════════════════════════════════╝
import asyncio, edge_tts

async def preview_segment(seg_id, override_emotion=None, override_voice=None):
    """Preview a single segment with optional overrides."""
    seg = next((s for s in enriched if s['id'] == seg_id), None)
    if not seg:
        print(f'Segment {seg_id} not found')
        return

    if override_emotion:
        rate, pitch, vol = resolve_prosody(override_emotion, seg['character'])
        seg = {**seg, 'rate': rate, 'pitch': pitch, 'volume': vol, 'emotion': override_emotion}
    if override_voice:
        seg = {**seg, 'voice': override_voice}

    print(f'▶ Segment {seg_id}: {seg["character"]} | {seg["emotion"]}')
    print(f'  Voice: {seg["voice"]}  rate={seg["rate"]}  pitch={seg["pitch"]}')
    print(f'  Text: {seg["text"][:100]}')

    out = f'/content/preview_seg_{seg_id}.mp3'
    comm = edge_tts.Communicate(
        seg['text'], seg['voice'],
        rate=seg['rate'], pitch=seg['pitch'], volume=seg['volume']
    )
    await comm.save(out)
    from IPython.display import Audio, display
    display(Audio(out, autoplay=True))

# ── Usage examples (uncomment to use): ────────────────────────────
# Preview segment 3 as-is:
# await preview_segment(3)

# Preview segment 3 with a different emotion:
# await preview_segment(3, override_emotion='scared')

# Preview with male voice:
# await preview_segment(3, override_voice='hi-IN-MadhurNeural')

# List all segments with their voice/emotion assignments:
print('📋 All segments:')
print(f'{"ID":<5} {"CHARACTER":<16} {"EMOTION":<18} {"TEXT[:50]"}')
print('-'*90)
for s in enriched:
    print(f'{s["id"]:<5} {s["character"]:<16} {s["emotion"]:<18} {s["text"][:50]}')